In [ ]:
#!pip3 install sklearn # or pip
#!pip3 install numpy
#!pip3 install torch torchvision

!pip install tensorboard
!pip install tensorflow-cpu

In [ ]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/AI

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/AI


In [ ]:
import numpy as np
import torch as t
from   matplotlib import pyplot as plt
import torch.nn.functional as F
import torch.nn as nn
from   torch.autograd import Variable
from   torch.utils.tensorboard import SummaryWriter

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler

iris = load_iris()
X = iris['data']
y = iris['target']
print("Sameple x", X[:3])
print("Sample y", y[100:103])
names  = iris['target_names']
feature_names = iris['feature_names']
print("Labels iris ", names)
print("Feature names", feature_names)
# Scale data to have mean 0 and variance 1
# which is importance for convergence of the neural network
# removes mean and divides by standard deviation
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Sameple x scaled\n", X_scaled[:3])

# Split the data set into training and testing
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size = 0.2, shuffle = True, random_state=2)

Sameple x [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]
Sample y [2 2 2]
Labels iris  ['setosa' 'versicolor' 'virginica']
Feature names ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Sameple x scaled
 [[-0.90068117  1.01900435 -1.34022653 -1.3154443 ]
 [-1.14301691 -0.13197948 -1.34022653 -1.3154443 ]
 [-1.38535265  0.32841405 -1.39706395 -1.3154443 ]]


In [ ]:
print(X_scaled.shape, X_train.shape, X_val.shape)
print(X_scaled[:3])
print(y[:3])

(150, 4) (120, 4) (30, 4)
[[-0.90068117  1.01900435 -1.34022653 -1.3154443 ]
 [-1.14301691 -0.13197948 -1.34022653 -1.3154443 ]
 [-1.38535265  0.32841405 -1.39706395 -1.3154443 ]]
[0 0 0]


In [ ]:
# convert data from numpy to tensors
X_t_train = t.from_numpy(X_train).float()
y_t_train = t.flatten(t.from_numpy(y_train).long()) # flatten - creates a one dimensional tensor
X_t_val   = t.from_numpy(X_val).float()
y_t_val   = t.flatten(t.from_numpy(y_val).long())

print(X_t_train.shape, y_t_train.shape)
print(X_t_train.dtype, y_t_train.dtype)
print(X_t_val.shape,   y_t_val.shape)
print(X_t_val.dtype,   y_t_val.dtype)

torch.Size([120, 4]) torch.Size([120])
torch.float32 torch.int64
torch.Size([30, 4]) torch.Size([30])
torch.float32 torch.int64


In [ ]:
# DataLoader = a class that shuffles the data and splits in into batches
# you should use it during training (SGD - accumulate error over batches of data )
train_data = [(X_t_train[i], y_t_train[i]) for i in range(X_t_train.shape[0])]
print("Sample train_data = ", train_data[:3], " type = ", type(train_data))
trainloader = t.utils.data.DataLoader(train_data, batch_size = 16, shuffle=True)

test_data = [(X_t_val[i], y_t_val[i]) for i in range(X_t_val.shape[0])]
testloader = t.utils.data.DataLoader(test_data, batch_size = 8)
#for x,label in trainloader:  # shuffles the data
#    print(x,label)

Sample train_data =  [(tensor([ 0.4322, -0.5924,  0.5922,  0.7907]), tensor(2)), (tensor([-0.9007,  0.5586, -1.1697, -0.9205]), tensor(0)), (tensor([-0.2948, -0.3622, -0.0898,  0.1325]), tensor(1))]  type =  <class 'list'>


In [ ]:
class Model(nn.Module):
    def __init__(self, input_dim = 4):

        super(Model, self).__init__()
        self.layer1    = nn.Linear(in_features = input_dim, out_features = 15)
        #self.dropout1 = nn.Dropout(p = 0.3) # drop 30% of output nodes from the previous layer during training only
        self.layer2    = nn.Linear(in_features= 15, out_features = 12)
        #self.dropout2 = nn.Dropout(p = 0.25)
        self.layer3    = nn.Linear(in_features = 12, out_features = 3) # 3 neurons = one for each class


    def forward(self, x):
        x = F.relu(self.layer1(x)) # activation function is RELU (rectified linear unit)
        #x = self.dropout1(x)
        x = F.relu(self.layer2(x))
        #x = self.dropout2(x)
        x = self.layer3(x) # linear outout neurons - output can be negative/positive
        return x


In [ ]:
model     = Model(X_train.shape[1])   # X_train.shape[1]
optimizer = t.optim.Adam(model.parameters(), lr=0.001)  #SGD = stochastic gradient descent
loss_fn   = nn.CrossEntropyLoss() # loss function = CrossEntropyLoss
print(model)

y0 = model.forward(X_t_train[1,:])
print(y0)

writer = SummaryWriter("runs/iris") # on the google drive

Model(
  (layer1): Linear(in_features=4, out_features=15, bias=True)
  (layer2): Linear(in_features=15, out_features=12, bias=True)
  (layer3): Linear(in_features=12, out_features=3, bias=True)
)
tensor([ 0.0217, -0.0078, -0.1271], grad_fn=<ViewBackward0>)


In [ ]:
# run this after the cell below

train_model(n_epochs = 100, model = model,
            train_loader = trainloader, test_loader = testloader, optimizer = optimizer,
            loss_fn = loss_fn,
            x_val   = X_t_val,   y_val = y_t_val)

writer.flush() # make sure that all pending events have been written to disk.

Epoch 0, Training loss 1.0590,  Validation loss 1.0393
Epoch 1, Training loss 1.0410,  Validation loss 1.0021
Epoch 5, Training loss 0.9434,  Validation loss 0.8455
Epoch 10, Training loss 0.7484,  Validation loss 0.5902
Epoch 15, Training loss 0.5867,  Validation loss 0.3954
Epoch 20, Training loss 0.5004,  Validation loss 0.3121
Epoch 25, Training loss 0.4409,  Validation loss 0.2779
Epoch 30, Training loss 0.3970,  Validation loss 0.2536
Epoch 35, Training loss 0.3521,  Validation loss 0.2308
Epoch 40, Training loss 0.3015,  Validation loss 0.2034
Epoch 45, Training loss 0.2826,  Validation loss 0.1761
Epoch 50, Training loss 0.2165,  Validation loss 0.1430
Epoch 55, Training loss 0.1867,  Validation loss 0.1202
Epoch 60, Training loss 0.1506,  Validation loss 0.1013
Epoch 65, Training loss 0.1246,  Validation loss 0.0881
Epoch 70, Training loss 0.1036,  Validation loss 0.0811
Epoch 75, Training loss 0.0896,  Validation loss 0.0757
Epoch 80, Training loss 0.0853,  Validation loss 0.

In [ ]:
writer.close()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=runs # --bind_all --load_fast false # start tensorboard

In [ ]:
def train_model(n_epochs, model, train_loader, test_loader, optimizer, loss_fn,
                x_val, y_val):
    last_val_loss = 100
    for epoch in range(n_epochs+1):
      model.train()       # set model in training mode = with dropout
      running_loss = 0.0
      correct = 0
      total = 0
      for xb,yb in train_loader: # for each batch
        optimizer.zero_grad() # set gradients to 0

        ym   = model.forward(xb)
        loss = loss_fn(ym,yb) # compute the loss between ym = output of the model and yb = correct output
        loss.backward()       # compute the gradients

        # adds the weights of each later in the tensorboard
        for name, param in model.named_parameters():
          if param.grad is not None:
            writer.add_histogram(f"{name}.grad", param.grad, epoch)

        optimizer.step()      # adjust the weights according to the gradients and the learning rate

        # compute loss, predicted, total,
        running_loss += loss.item()
        _, predicted = t.max(ym.data, 1)
        total   += yb.size(0)
        correct += (predicted == yb).sum().item()

      acc        = 100 * correct / total # accuracy
      train_loss = running_loss / len(train_loader)

      # 🧠 Log weights after update
      for name, param in model.named_parameters():
        writer.add_histogram(f"{name}.weight", param.data, epoch)

      writer.add_scalar("Loss/train", train_loss, epoch) # training loss
      writer.add_scalar("Accuracy/train", acc, epoch)

      if epoch == 1 or epoch % 5 == 0:
        model.eval() # set model in evaluation mode = no dropout

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with t.no_grad():  # no learning
          for xb,yb in test_loader: # for each batch
            ym_val   = model.forward(xb)
            loss_val = loss_fn(ym_val,yb)

            val_running_loss += loss_val
            _, predicted = t.max(ym_val.data, 1)
            val_total   += yb.size(0)
            val_correct += (predicted == yb).sum().item()

          val_acc        = 100 * val_correct / val_total # accuracy
          val_loss = val_running_loss / len(test_loader)

          writer.add_scalar("Loss/val", val_loss, epoch) # training loss
          writer.add_scalar("Accuracy/val", val_acc, epoch)

          print(f"Epoch {epoch}, Training loss {train_loss:.4f},", end = " ")
          print(f" Validation loss {loss_val.item():.4f}")
          #if last_val_loss < loss_val:
          #  print("Early stopping at epoch", epoch)
          #  return
          #last_val_loss = loss_val
    print("Training complete.")

In [ ]:
soft_max  = t.nn.Softmax(1)
y_m_train = soft_max(model.forward(X_t_train)) # y_m normalized with softmax
y_m_train = t.argmax(y_m_train,dim = 1)

y_m_val   = soft_max(model.forward(X_t_val))
y_m_val   = t.argmax(y_m_val,dim = 1)
correct_pred_val = t.sum(y_m_val == y_t_val)/y_m_val.shape[0]
print("Validation accuracy = ", correct_pred_val)

correct_pred_train = t.sum(y_m_train == y_t_train)/y_m_train.shape[0]
print("Training accuracy = ", correct_pred_train)

Validation accuracy =  tensor(0.9667)
Training accuracy =  tensor(0.9833)
